<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

class AbsMagnetization(qml.measurements.StateMeasurement):
    """A measurement class that estimates <|M|>."""

    def process_state(self, state, wire_order):
        """
        Calculates <|M|> for the input quantum state.
        Args:
            state (Sequence[complex]): quantum state with flat shape.
            wire_order (Wires): wires determining the subspace.
        Returns:
            abs_mag (float): <|M|>
        """
        state = qml.state().process_state(state, wire_order)
        num_qubits = len(wire_order)
        k = 2 ** num_qubits

        probs = np.abs(state) ** 2

        abs_mags = []
        for idx in range(k):
            bin_state = f"{idx:0{num_qubits}b}"
            # '0' is +1, '1' is -1 for PauliZ
            mag = 0
            for s in bin_state:
                mag += 1 if s == '0' else -1
            abs_mags.append(np.abs(mag))
        abs_mags = np.array(abs_mags, dtype=float)
        exp = np.sum(probs * abs_mags)
        abs_mag = exp / num_qubits
        return abs_mag

def tfim_ground_state(num_qubits, h):
    """Calculates the ground state of the 1D TFIM Hamiltonian with open boundary conditions."""
    ops = []
    coeffs = []
    wires = list(range(num_qubits))

    # Nearest-neighbor in a 1D chain (open boundary): i=0 to N-2, link i and i+1
    for i in range(num_qubits - 1):
        ops.append(qml.PauliZ(i) @ qml.PauliZ(i + 1))
        coeffs.append(-1.0)
    # Transverse field terms
    for i in range(num_qubits):
        ops.append(qml.PauliX(i))
        coeffs.append(-h)

    H = qml.Hamiltonian(coeffs, ops)
    H_mat = qml.matrix(H)
    evals, evecs = np.linalg.eigh(H_mat)
    groundstate = evecs[:, np.argmin(evals)]
    return np.array(groundstate)

def make_magnetization_qnode(num_qubits):
    dev = qml.device("default.qubit", wires=num_qubits)
    @qml.qnode(dev)
    def magnetization(h):
        groundstate = tfim_ground_state(num_qubits, h)
        qml.StatePrep(groundstate, wires=list(range(num_qubits)))
        return AbsMagnetization(wires=list(range(num_qubits)))
    return magnetization

def critical_point_estimate(mags, h_values):
    """Estimate the critical point as the field value where magnetization crosses 0.5, using linear interpolation."""
    mags = np.array(mags)
    h_values = np.array(h_values)
    # The typical behavior is mags is monotonic decreasing (from 1 to 0) as h increases
    for i in range(len(mags)-1):
        m0, m1 = mags[i], mags[i+1]
        h0, h1 = h_values[i], h_values[i+1]
        if (m0 >= 0.5 and m1 < 0.5) or (m0 <= 0.5 and m1 > 0.5):
            # Linear interpolation
            if m1 != m0:
                h_cross = h0 + (0.5 - m0) * (h1 - h0) / (m1 - m0)
                return float(h_cross)
            else:
                return float((h0 + h1) / 2)
    # Fallback: choose closest value
    idx = np.argmin(np.abs(mags - 0.5))
    return float(h_values[idx])

def run(test_case_input: str) -> str:
    num_qubits = json.loads(test_case_input)
    h_values = np.arange(0.2, 1.1, 0.005)
    mags = []
    magnetization = make_magnetization_qnode(num_qubits)
    for h in h_values:
        mags.append(magnetization(h))
    output = critical_point_estimate(np.array(mags), h_values)
    # Output must be rounded to 3 decimals, as [h_c] in JSON array
    return json.dumps([round(float(output), 3)])

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.isclose(solution_output, expected_output, rtol=5e-3)

# TEST CASES
def test():
    for input_, expected_output in [('5', '[0.633]'), ('2', '[0.353]')]:
        output = run(input_)
        check(output, expected_output)
test()


AssertionError: 